In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from google import genai
gemini_client = genai.Client() # picks up the API key from the env variable GEMINI_API_KEY

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
assistant = RAGBase(
    index = index,
    llm_client=gemini_client
)

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
from google.genai import types

# Define the schema explicitly
search_declaration = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
    }
)

search_tool = types.Tool(
    function_declarations=[search_declaration]
)

*
*
*
Conversation History

1. Making a request (query) to the LLM <-- first request
2. LLM decides to invoke Search('with parameters')
3. Getting results as Search() output
4. Sending the results back to the LLM <-- a second request
5. LLM processes the results
6. LLM gives an answer

In [ ]:
agent_instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [11]:
from google.genai import types

def make_call(call):
    # Gemini already parsed the args into a dict!
    args = call.args

    if call.name == "search":
        result = search(**args)

    # Gemini expects a specific Content/Part object back
    return types.Content(
        role="user",
        parts=[
            types.Part.from_function_response(
                name=call.name,
                response={"result": result}
            )
        ]
    )

In [ ]:
from google.genai import types

def agent_loop(instructions, question, model="gemini-3.6-flash") -> str:

    conversation_history = [
        {'role': 'user', 'parts': [{'text': question}]},
    ]

    conv_iteration = 1
    while True:
        print(f"iteration #{conv_iteration}...")
        has_function_calls = False

        # Call the model with the current conversation history
        response = gemini_client.models.generate_content(
            model = model,
            contents=conversation_history,
            config=types.GenerateContentConfig(
                system_instruction = instructions,
                tools=[search_tool]
            )
        )

        # Append the model's response to the history
        conversation_history.append(response.candidates[0].content)

        for part in response.candidates[0].content.parts:
            
            # Checking if the item is a function call
            if part.function_call:
                print("function_call:", part.function_call.name, part.function_call.args)
                call_output = make_call(part.function_call)
                conversation_history.append(call_output)
                has_function_calls = True

            # Checking if the item is a standard message
            elif part.text:
                print("ASSISTANT:")
                last_answer = part.text
                print(last_answer)

        conv_iteration += 1

        # Exit condition
        if has_function_calls == False:
            break

    return last_answer

iteration #1...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 3.791777541s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.6-flash', 'location': 'global'}, 

In [ ]:
agent_loop(agent_instructions, "what's queen gambit?")

In [ ]:
""" def calculate_gemini_flash_price(response):

    usage = response.usage_metadata
    input_tokens = usage.prompt_token_count
    output_tokens = usage.candidates_token_count

    print("Input Tokens:", input_tokens)
    print("Output Tokens:", output_tokens)

    # Standard pricing rates for gemini-3.6-flash (per million tokens)
    INPUT_PRICE_PER_MILLION = 0.75
    OUTPUT_PRICE_PER_MILLION = 4.50

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost


    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost:": total_cost,
    }


cost_info = calculate_gemini_flash_price(response)
print("Total cost: $", round(cost_info["total_cost"], 8)) """

Input Tokens: 76
Output Tokens: 19
Total cost: $ 0.0001425


In [ ]:
agent_loop(agent_instructions, "I just discovered the course. Can I join it?")